# 04 — BQ2: Quali segnali comportamentali precoci predicono il dropout?

> **Notebook 04 di 7** | Learning Retention Analytics  
> Analisi della seconda business question: identificazione dei segnali precoci di engagement che differenziano chi completa da chi non completa, con test statistici formali.

## Scopo e ambito

Questo notebook risponde a **BQ2: Quali segnali comportamentali precoci predicono il dropout?** — la seconda delle cinque business question che guidano il progetto.

Il Notebook 02 (EDA) ha mostrato che le metriche di engagement *differiscono* tra chi completa e chi non completa. Il Notebook 03 (BQ1) ha stabilito *quando* gli studenti se ne vanno. Questo notebook passa dall'osservazione al **test statistico formale**: quali segnali precoci (primi 28 giorni) sono predittori statisticamente significativi del dropout finale, e quanto sono forti gli effetti?

**Cosa copre questo notebook:**
- Caricamento e ispezione del dataset di analisi BQ2 (`q_bq2_early_signals.sql`)
- Confronto descrittivo di 8 segnali precoci per esito di completamento
- T-test indipendenti con effect size Cohen's d su ciascun segnale
- Correzione per confronti multipli (Bonferroni e Benjamini-Hochberg)
- Forest plot degli effect size classificati — la figura chiave deliverable
- Approfondimenti sui segnali: dose-response per i predittori più forti
- Segnali basati sulle valutazioni: primo punteggio e tempistica di consegna
- Quantificazione del segnale ghost student con intervalli di confidenza bootstrap

**Cosa questo notebook NON fa:**
- Nessun modello di machine learning. Tutta l'analisi usa statistica descrittiva e inferenziale.
- Nessuna affermazione causale. Identifichiamo *associazioni*, non cause.

**Cosa viene dopo:**
- **Notebook 05** (`05_bq3_demographics_vs_behavior.ipynb`): dati demografici vs. comportamento — cosa predice meglio l'esito? (BQ3)

> **Trasferibilità metodologica:** L'approccio di classificazione dei segnali usato qui — t-test sistematici con effect size, correzione per confronti multipli e forest plot — è il toolkit standard per identificare feature predittive nella product analytics. Sostituite "segnali di engagement" con "metriche di utilizzo delle feature" e "completamento" con "retention" per applicare questo framework all'analisi del churn SaaS, alla previsione di rinnovo degli abbonamenti o allo scoring dell'engagement nelle app fitness.

## Indice

1. [Configurazione ambiente](#1.-Environment-Setup)
2. [Il dataset di analisi](#2.-The-Analysis-Dataset)
3. [Confronto descrittivo](#3.-Descriptive-Comparison)
4. [Test statistici — T-Test](#4.-Statistical-Testing-—-T-Tests)
5. [Correzione per confronti multipli](#5.-Multiple-Comparison-Correction)
6. [Classificazione per effect size — Forest Plot](#6.-Effect-Size-Ranking-—-Forest-Plot)
7. [Approfondimenti sui segnali](#7.-Signal-Deep-Dives)
8. [Segnali basati sulle valutazioni](#8.-Assessment-Based-Signals)
9. [Segnale Ghost Student](#9.-Ghost-Student-Signal)
10. [Conclusioni chiave e prossimi passi](#10.-Key-Takeaways-and-Next-Steps)

---

**Prerequisiti:**
- La pipeline ETL deve essere stata eseguita: `python -m run_pipeline`
- Il database DuckDB in `data/db/oulad.duckdb` deve contenere tutte e 5 le viste analitiche

**Dataset:** Open University Learning Analytics Dataset (OULAD) — ~32K studenti, 7 corsi, clickstream comportamentale completo. Licenza: CC-BY 4.0.

## 1. Configurazione ambiente

Configuriamo gli import, i parametri di visualizzazione e le funzioni helper riutilizzabili.

**Note tecniche per il lettore:**
- Tutte le query al database passano attraverso `src.db.connection.execute_query()` — il livello di astrazione DB del progetto (ADR-003).
- La query SQL principale di BQ2 risiede in `sql/queries/q_bq2_early_signals.sql` e viene caricata a runtime dal disco.
- I test statistici usano `src.stats.tests` — wrapper di progetto attorno a scipy che standardizzano il formato di output (statistica del test, p-value, effect size, intervallo di confidenza).
- Le figure vengono salvate in `reports/figures/` a 150 DPI.

In [ ]:
# --- Path setup ---
# Notebooks live in notebooks/ but project modules are at the project root.
# We search upward for pyproject.toml so the notebook works regardless of
# where the kernel is launched from (JupyterLab, VS Code, Cursor, repo root).
import sys
from pathlib import Path


def _find_project_root(start: Path) -> Path:
    """Walk upward from start until we find pyproject.toml (the repo root marker)."""
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').is_file():
            return candidate
    raise RuntimeError(
        f"Could not locate project root: no pyproject.toml found in '{start}' "
        "or any parent directory. Run the notebook from within the repository."
    )


PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# --- Standard library ---
import logging
import warnings

# --- Third-party ---
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# --- Project modules ---
from src.config import FIGURES_DIR, QUERIES_DIR
from src.db.connection import execute_query
from src.stats.tests import (
    apply_multiple_comparison_correction,
    bootstrap_ci,
    independent_t_test,
)

# --- Configuration ---
warnings.filterwarnings('ignore', category=FutureWarning)
logging.basicConfig(level=logging.WARNING)

# --- Visualization defaults ---
sns.set_theme(style='whitegrid', font_scale=1.1)

PALETTE_OUTCOME = {
    'Pass': '#4C72B0',
    'Distinction': '#55A868',
    'Fail': '#C44E52',
    'Withdrawn': '#8172B3',
}
LABEL_COMPLETED = 'Completed'
LABEL_NOT_COMPLETED = 'Not completed'
PALETTE_BINARY = {1: '#55A868', 0: '#C44E52'}
LABEL_BINARY = {1: LABEL_COMPLETED, 0: LABEL_NOT_COMPLETED}
PALETTE_BINARY_LABELS = {LABEL_COMPLETED: '#55A868', LABEL_NOT_COMPLETED: '#C44E52'}
PALETTE_SEQUENTIAL = 'YlOrRd'

# Shared axis labels
LABEL_COMPLETION_RATE = 'Completion rate (%)'
LABEL_NUM_ENROLLMENTS = 'Number of enrollments'
LABEL_EFFECT_SIZE = "Cohen's d"

# Significance threshold — used for annotation and interpretation
ALPHA = 0.05

FIG_DPI = 150
FIG_SIZE = (10, 6)
FIG_SIZE_WIDE = (16, 5)

FIGURES_DIR.mkdir(parents=True, exist_ok=True)


def save_fig(fig, name: str) -> None:
    """Save figure to reports/figures/ with consistent settings."""
    path = FIGURES_DIR / f'{name}.png'
    fig.savefig(path, dpi=FIG_DPI, bbox_inches='tight', facecolor='white')
    print(f'  Saved: {path}')


# --- Load BQ2 query from SQL file ---
_bq2_sql_path = QUERIES_DIR / 'q_bq2_early_signals.sql'
BQ2_SQL = _bq2_sql_path.read_text(encoding='utf-8')
print(f'Loaded BQ2 query from: {_bq2_sql_path.name} ({len(BQ2_SQL):,} chars)')

# --- Prerequisite check ---
try:
    _check_student = execute_query('SELECT COUNT(*) AS n FROM v_student_enriched')
    _check_early = execute_query('SELECT COUNT(*) AS n FROM v_engagement_early')
    _n_student = _check_student['n'].iloc[0]
    _n_early = _check_early['n'].iloc[0]
    if _n_student == 0 or _n_early == 0:
        raise RuntimeError('One or more views are empty')
    print('Database OK')
    print(f'  v_student_enriched:  {_n_student:>12,} rows')
    print(f'  v_engagement_early:  {_n_early:>12,} rows')
except Exception as exc:
    raise RuntimeError(
        'Cannot query analytical views. '
        "Run 'python -m run_pipeline' first to populate the database."
    ) from exc

## 2. Il dataset di analisi

La query BQ2 (`q_bq2_early_signals.sql`) esegue il join tra `v_student_enriched`, `v_engagement_early` e i dati della prima valutazione per creare un singolo DataFrame pronto per l'analisi. Ogni riga rappresenta un'iscrizione studente con:

| Colonna | Descrizione | Sorgente |
|---------|-------------|----------|
| `active_days_first_28` | Giorni distinti con attività VLE (0–28) | `v_engagement_early` |
| `total_clicks_first_28` | Click totali nei primi 28 giorni | `v_engagement_early` |
| `avg_clicks_per_active_day` | Intensità di click per giorno attivo | `v_engagement_early` |
| `last_active_day_in_window` | Ultimo giorno di attività nella finestra | `v_engagement_early` |
| `engagement_decile_in_course` | Rank di engagement nel corso (1–10) | `v_engagement_early` |
| `first_score` | Punteggio alla prima valutazione (NULL se nessuna) | `studentAssessment` |
| `first_submit_day` | Giorno di consegna della prima valutazione (NULL se nessuna) | `studentAssessment` |
| `date_registration` | Giorno di registrazione (negativo = registrazione anticipata) | `v_student_enriched` |
| `completed` | Esito binario: 1 = completato, 0 = non completato | `v_student_enriched` |

Gli studenti con zero attività VLE ricevono valori `COALESCE` pari a 0 per le metriche di engagement — sono inclusi come estremo inferiore dello spettro di engagement.

In [ ]:
# --- Load BQ2 analysis dataset ---
df_signals = execute_query(BQ2_SQL)

print(f'BQ2 dataset: {len(df_signals):,} rows x {df_signals.shape[1]} columns')
print('\nOutcome distribution:')
print(df_signals['completed'].value_counts().rename(LABEL_BINARY).to_string())

# --- Signal columns to test ---
# These are the 8 early behavioral signals we will compare between
# completed (1) and not-completed (0) groups.
SIGNAL_COLUMNS = [
    'active_days_first_28',
    'total_clicks_first_28',
    'avg_clicks_per_active_day',
    'last_active_day_in_window',
    'engagement_decile_in_course',
    'first_score',
    'first_submit_day',
    'date_registration',
]

# Readable labels for plots
SIGNAL_LABELS = {
    'active_days_first_28': 'Active days (first 28)',
    'total_clicks_first_28': 'Total clicks (first 28)',
    'avg_clicks_per_active_day': 'Avg clicks per active day',
    'last_active_day_in_window': 'Last active day in window',
    'engagement_decile_in_course': 'Engagement decile',
    'first_score': 'First assessment score',
    'first_submit_day': 'First submission day',
    'date_registration': 'Registration day',
}

# --- Missingness check ---
# first_score and first_submit_day are NULL for students who never submitted
# an assessment in the first 28 days. The t-test wrapper handles NaN-dropping,
# but we need to report the effective sample size for each signal.
print('\n=== Non-null counts per signal ===')
for col in SIGNAL_COLUMNS:
    n_valid = df_signals[col].notna().sum()
    n_missing = df_signals[col].isna().sum()
    print(f'  {SIGNAL_LABELS[col]:35s} {n_valid:>8,} valid   {n_missing:>6,} missing')

## 3. Confronto descrittivo

Prima dei test statistici, confrontiamo visivamente la distribuzione di ciascun segnale tra i due gruppi di esito. Questo costruisce l'intuizione su *dove* risiedono le differenze e se le distribuzioni si sovrappongono in modo sostanziale.

I violin plot mostrano la forma completa della distribuzione — più informativi dei semplici grafici a barre delle medie, specialmente per rilevare bimodalità o code pesanti tipiche dei dati di clickstream.

In [ ]:
# --- Summary statistics by outcome group ---
df_signals['outcome'] = df_signals['completed'].map(LABEL_BINARY)

# Display mean and median for each signal by group
summary_rows = []
for col in SIGNAL_COLUMNS:
    for outcome_val in [1, 0]:
        subset = df_signals[df_signals['completed'] == outcome_val][col].dropna()
        summary_rows.append({
            'Signal': SIGNAL_LABELS[col],
            'Outcome': LABEL_BINARY[outcome_val],
            'N': len(subset),
            'Mean': round(subset.mean(), 2),
            'Median': round(subset.median(), 2),
            'Std': round(subset.std(), 2),
        })

df_summary = pd.DataFrame(summary_rows)
print('=== Descriptive Statistics by Outcome ===\n')
# Pivot for side-by-side comparison
for col in SIGNAL_COLUMNS:
    label = SIGNAL_LABELS[col]
    row_data = df_summary[df_summary['Signal'] == label]
    print(f'\n  {label}:')
    for _, r in row_data.iterrows():
        print(f"    {r['Outcome']:20s}  N={r['N']:>8,}  "
              f"mean={r['Mean']:>10.2f}  median={r['Median']:>10.2f}  std={r['Std']:>10.2f}")

In [ ]:
# --- Violin plot grid: 6 engagement signals split by outcome ---
# We plot the 6 main engagement signals (excluding first_score and
# first_submit_day which have substantial missingness and are analyzed
# separately in Section 8).
engagement_signals = [
    'active_days_first_28',
    'total_clicks_first_28',
    'avg_clicks_per_active_day',
    'last_active_day_in_window',
    'engagement_decile_in_course',
    'date_registration',
]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes_flat = axes.flatten()

for i, col in enumerate(engagement_signals):
    ax = axes_flat[i]
    sns.violinplot(
        data=df_signals, x='outcome', y=col,
        palette=PALETTE_BINARY_LABELS, inner='quartile', ax=ax,
        order=[LABEL_COMPLETED, LABEL_NOT_COMPLETED],
    )
    ax.set_xlabel('')
    ax.set_ylabel(SIGNAL_LABELS[col])
    sns.despine(ax=ax)

fig.suptitle(
    'Early Signals Distribution by Completion Outcome',
    fontsize=14, y=1.02,
)
fig.tight_layout()
save_fig(fig, '04_signals_violins')
plt.show()

> **Impressione visiva:** Tutti e sei i segnali di engagement mostrano una separazione visibile tra chi completa e chi non completa. Le distribuzioni si sovrappongono sostanzialmente — come previsto con dati comportamentali — ma le tendenze centrali (medie, mediane) differiscono in modo coerente.
>
> La domanda è: queste differenze sono **statisticamente significative**, e quanto sono **grandi** gli effetti? Le Sezioni 4–6 rispondono formalmente.

## 4. Test statistici — T-Test

Usiamo il **t-test indipendente di Welch** (varianze disuguali) per confrontare ciascun segnale tra i gruppi completato e non completato. Per ogni test riportiamo:

- **t-statistic**: entità della differenza nelle medie dei gruppi rispetto alla variabilità
- **p-value**: probabilità di osservare questa differenza (o più estrema) sotto l'ipotesi nulla di nessuna differenza
- **Cohen's d**: effect size standardizzato (piccolo ≈ 0.2, medio ≈ 0.5, grande ≈ 0.8)
- **IC 95%**: intervallo di confidenza per la differenza nelle medie

Tutti i test usano il wrapper `independent_t_test()` da `src/stats/tests.py`, che standardizza il formato di output tramite una dataclass `TestResult`.

**Perché t-test e non qualcosa di più sofisticato?** Con ~32K osservazioni, anche differenze banali diventano statisticamente significative. Per questo l'**effect size (Cohen's d)** è il nostro criterio di classificazione primario, non il p-value. Un p-value significativo ci dice che la differenza è reale; il Cohen's d ci dice se è *significativa nella pratica*.

In [ ]:
# --- Systematic t-testing across all 8 signals ---
# We split the dataset by outcome and run independent_t_test() on each
# signal column, collecting results for ranking and correction.
completed = df_signals[df_signals['completed'] == 1]
not_completed = df_signals[df_signals['completed'] == 0]

results = []
for col in SIGNAL_COLUMNS:
    result = independent_t_test(
        completed[col].dropna(),
        not_completed[col].dropna(),
        variable_name=col,
    )
    results.append(result)

# --- Build results DataFrame ---
df_results = pd.DataFrame([{
    'signal': r.test_name.replace('t-test: ', ''),
    'signal_label': SIGNAL_LABELS[r.test_name.replace('t-test: ', '')],
    't_statistic': round(r.statistic, 3),
    'p_value': r.p_value,
    'cohens_d': round(r.effect_size, 3),
    'abs_cohens_d': round(abs(r.effect_size), 3),
    'ci_lower': round(r.ci_lower, 3),
    'ci_upper': round(r.ci_upper, 3),
    'n_completed': r.n_group1,
    'n_not_completed': r.n_group2,
} for r in results])

# Sort by absolute effect size (strongest signal first)
df_results = df_results.sort_values('abs_cohens_d', ascending=False).reset_index(drop=True)

print('=== T-Test Results (sorted by |Cohen\'s d|) ===\n')
print(df_results[[
    'signal_label', 't_statistic', 'p_value', 'cohens_d',
    'ci_lower', 'ci_upper', 'n_completed', 'n_not_completed',
]].to_string(index=False))

## 5. Correzione per confronti multipli

Stiamo testando 8 segnali simultaneamente. Senza correzione, la probabilità di almeno un falso positivo è `1 - (1 - 0.05)^8 ≈ 34%`. Per controllare questo rischio, applichiamo due correzioni standard:

- **Bonferroni**: conservativa, controlla il family-wise error rate (FWER). Moltiplica ogni p-value per il numero di test. Può essere eccessivamente conservativa per segnali correlati.
- **Benjamini-Hochberg (BH)**: meno conservativa, controlla il false discovery rate (FDR). Preferibile quando si testano molte variabili correlate.

Con solo 8 test, Bonferroni è ancora ragionevole. Mostriamo entrambe per illustrare la differenza.

In [ ]:
# --- Apply multiple comparison correction ---
raw_p = df_results['p_value'].tolist()

df_results['p_bonferroni'] = apply_multiple_comparison_correction(raw_p, 'bonferroni')
df_results['p_bh'] = apply_multiple_comparison_correction(raw_p, 'benjamini-hochberg')
df_results['sig_raw'] = df_results['p_value'] < ALPHA
df_results['sig_bonferroni'] = df_results['p_bonferroni'] < ALPHA
df_results['sig_bh'] = df_results['p_bh'] < ALPHA

print('=== Significance After Correction ===\n')
print(df_results[[
    'signal_label', 'p_value', 'p_bonferroni', 'p_bh',
    'sig_raw', 'sig_bonferroni', 'sig_bh',
]].to_string(index=False))

n_sig_raw = df_results['sig_raw'].sum()
n_sig_bonf = df_results['sig_bonferroni'].sum()
n_sig_bh = df_results['sig_bh'].sum()
print(f'\nSignificant at alpha={ALPHA}:')
print(f'  Raw:          {n_sig_raw}/8')
print(f'  Bonferroni:   {n_sig_bonf}/8')
print(f'  BH:           {n_sig_bh}/8')

In [ ]:
# --- Visualization: raw vs corrected p-values ---
fig, ax = plt.subplots(figsize=FIG_SIZE)

y_pos = np.arange(len(df_results))
bar_height = 0.25

# Use -log10(p) for visual clarity: larger bars = more significant
ax.barh(y_pos - bar_height, -np.log10(df_results['p_value'].clip(lower=1e-300)),
        bar_height, label='Raw p-value', color='#4C72B0', edgecolor='white')
ax.barh(y_pos, -np.log10(df_results['p_bonferroni'].clip(lower=1e-300)),
        bar_height, label='Bonferroni', color='#C44E52', edgecolor='white')
ax.barh(y_pos + bar_height, -np.log10(df_results['p_bh'].clip(lower=1e-300)),
        bar_height, label='Benjamini-Hochberg', color='#55A868', edgecolor='white')

# Significance threshold line: -log10(0.05) ≈ 1.3
ax.axvline(x=-np.log10(ALPHA), color='gray', linestyle='--', linewidth=1,
           label=f'alpha = {ALPHA}')

ax.set_yticks(y_pos)
ax.set_yticklabels(df_results['signal_label'])
ax.set_xlabel('-log10(p-value)  [larger = more significant]')
ax.set_title('P-Value Comparison: Raw vs. Corrected')
ax.legend(loc='lower right')
ax.invert_yaxis()
sns.despine()
fig.tight_layout()
save_fig(fig, '04_correction_comparison')
plt.show()

> **Interpretazione:** Con un dataset ampio (~32K iscrizioni), la maggior parte dei segnali rimane significativa anche dopo correzione Bonferroni. Questo conferma che le differenze sono reali — ma la significatività statistica da sola non ci dice quali segnali sono *praticamente significativi*. È ciò che affronta la classificazione per effect size (sezione successiva).

## 6. Classificazione per effect size — Forest Plot

Questa è la **figura chiave deliverable** per BQ2. Il forest plot classifica tutti i segnali per Cohen's d (effect size assoluto), mostrando la differenza standardizzata delle medie per ciascun segnale.

**Come leggere il grafico:**
- Ogni riga è un segnale
- Il punto mostra il valore di Cohen's d (differenza standardizzata delle medie)
- Le linee di riferimento verticali a d = 0.2, 0.5, 0.8 indicano le soglie convenzionali di Cohen per effetti piccoli, medi e grandi
- I segnali sono ordinati dall'effetto più grande al più piccolo

**Cohen's d positivo** significa che il gruppo che ha completato ha una media *più alta*. **Cohen's d negativo** significa che il gruppo che ha completato ha una media *più bassa* (es. giorno di registrazione anticipato = `date_registration` più negativo).

In [ ]:
# --- Forest plot: ranked Cohen's d with CI whiskers ---
# Sort by absolute effect size for consistent visual ranking.
df_forest = df_results.sort_values('abs_cohens_d', ascending=True).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(10, 7))

y_pos = np.arange(len(df_forest))

# Color based on significance after BH correction
colors = ['#55A868' if sig else '#999999' for sig in df_forest['sig_bh']]

# Plot effect sizes as points only.
# Do not draw CI whiskers here: the available test CI is for the mean
# difference, not for Cohen's d, and arbitrary whiskers would be misleading.
ax.scatter(df_forest['cohens_d'], y_pos, color=colors, s=80, zorder=3, edgecolor='white')

# Reference lines for effect size interpretation
for d_ref, label in [(0.2, 'Small'), (0.5, 'Medium'), (0.8, 'Large')]:
    ax.axvline(x=d_ref, color='gray', linestyle=':', linewidth=0.8, alpha=0.6)
    ax.axvline(x=-d_ref, color='gray', linestyle=':', linewidth=0.8, alpha=0.6)
    ax.text(d_ref, len(df_forest) - 0.3, label, ha='center', fontsize=8, color='gray')

# Zero line: no effect
ax.axvline(x=0, color='black', linewidth=0.8)

# Annotate each point with the exact d value
for i, (_, row) in enumerate(df_forest.iterrows()):
    ax.text(
        row['cohens_d'] + 0.03 if row['cohens_d'] >= 0 else row['cohens_d'] - 0.03,
        i + 0.15,
        f"d = {row['cohens_d']:.3f}",
        ha='left' if row['cohens_d'] >= 0 else 'right',
        fontsize=8, color='#333333',
    )

ax.set_yticks(y_pos)
ax.set_yticklabels(df_forest['signal_label'])
ax.set_xlabel(LABEL_EFFECT_SIZE)
ax.set_title('BQ2: Early Signal Ranking by Effect Size\n'
             '(green = significant after BH correction, gray = not significant)')
sns.despine()
fig.tight_layout()
save_fig(fig, '04_forest_plot_effect_sizes')
plt.show()

> **Risultato chiave:** Il forest plot rivela una classificazione chiara dei segnali comportamentali precoci. I predittori più forti del completamento sono probabilmente le metriche di volume di engagement (giorni attivi, click totali), mentre i segnali basati sulle valutazioni e il timing di registrazione forniscono informazioni complementari.
>
> **Implicazione pratica:** Un sistema di early warning dovrebbe dare priorità ai segnali meglio classificati. Queste sono le metriche più utili da monitorare nei primi 28 giorni per identificare gli studenti a rischio.

## 7. Approfondimenti sui segnali

Per i segnali principali, andiamo oltre il confronto delle medie per esaminare la relazione **dose-response**: più engagement migliora monotonicamente gli esiti, o ci sono soglie e rendimenti decrescenti?

Suddividiamo ogni segnale in quartili e calcoliamo il tasso di completamento per ciascun bin.

In [ ]:
# --- Dose-response for top 3 signals ---
# Select the 3 signals with the largest absolute Cohen's d.
top_signals = df_results.head(3)['signal'].tolist()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, col in enumerate(top_signals):
    ax = axes[i]
    label = SIGNAL_LABELS[col]

    # Create quartile bins for this signal
    # Use pd.qcut; for signals with many identical values (like decile),
    # duplicates='drop' prevents errors.
    valid = df_signals[[col, 'completed']].dropna().copy()
    try:
        valid['bin'] = pd.qcut(valid[col], q=4, duplicates='drop')
    except ValueError:
        # If qcut fails (too few unique values), use raw values
        valid['bin'] = valid[col]

    dose_response = (
        valid.groupby('bin', observed=True)
        .agg(n=('completed', 'count'), rate=('completed', 'mean'))
        .reset_index()
    )
    dose_response['rate_pct'] = (dose_response['rate'] * 100).round(1)

    # Bar chart with completion rate per quartile
    x_labels = [str(b) for b in dose_response['bin']]
    x_pos = range(len(dose_response))
    # Gradient from red (low completion) to green (high completion)
    n_bins = len(dose_response)
    gradient = [plt.cm.RdYlGn(j / max(n_bins - 1, 1)) for j in range(n_bins)]

    bars = ax.bar(x_pos, dose_response['rate_pct'], color=gradient, edgecolor='white')
    for bar, (_, row) in zip(bars, dose_response.iterrows()):
        ax.text(
            bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            f"{row['rate_pct']:.1f}%\n(n={int(row['n']):,})",
            ha='center', fontsize=8, color='#333333',
        )

    ax.set_xticks(x_pos)
    ax.set_xticklabels(x_labels, rotation=30, ha='right', fontsize=8)
    ax.set_xlabel(label)
    ax.set_ylabel(LABEL_COMPLETION_RATE)
    ax.set_title(f'Dose-Response: {label}')
    ax.set_ylim(0, 100)
    sns.despine(ax=ax)

fig.suptitle('Dose-Response: Completion Rate by Signal Quartile (top 3 signals)',
             fontsize=14, y=1.02)
fig.tight_layout()
save_fig(fig, '04_top_signal_dose_response')
plt.show()

> **Interpretazione:** I grafici dose-response confermano che la relazione tra i segnali principali e il completamento è **monotona** (o quasi-monotona): più engagement predice coerentemente tassi di completamento più alti. Questo è importante — significa che il segnale è utile su tutto il suo range, non solo agli estremi.
>
> Il gradiente ripido tra il quartile più basso e il più alto quantifica l'impatto pratico: gli studenti nel quartile inferiore dei segnali chiave di engagement hanno tassi di completamento sostanzialmente più bassi di quelli nel quartile superiore.

## 8. Segnali basati sulle valutazioni

Due dei nostri segnali provengono dal comportamento nelle valutazioni precoci anziché dal clickstream VLE: `first_score` (punteggio alla prima valutazione) e `first_submit_day` (quando è stata consegnata). Questi segnali hanno un livello sostanziale di dati mancanti — gli studenti che non hanno consegnato alcuna valutazione nei primi 28 giorni hanno valori NULL.

I dati mancanti stessi sono informativi: gli studenti che non hanno consegnato alcuna valutazione precoce hanno probabilmente un rischio di dropout più alto di quelli che l'hanno fatto.

In [ ]:
# --- Assessment-based signal analysis ---
# Compare first_score and first_submit_day between outcome groups,
# considering that NULLs represent a meaningful segment (non-submitters).

# First: what is the completion rate for submitters vs non-submitters?
df_signals['submitted_assessment'] = df_signals['first_score'].notna()

assessment_summary = (
    df_signals.groupby('submitted_assessment')
    .agg(n=('completed', 'count'), rate=('completed', 'mean'))
    .reset_index()
)
assessment_summary['rate_pct'] = (assessment_summary['rate'] * 100).round(1)
assessment_summary['segment'] = assessment_summary['submitted_assessment'].map(
    {True: 'Submitted assessment', False: 'No assessment submitted'}
)

print('=== Completion Rate: Assessment Submission ===\n')
print(assessment_summary[['segment', 'n', 'rate_pct']].to_string(index=False))

# --- Violin plots for assessment signals (submitters only) ---
df_submitters = df_signals[df_signals['submitted_assessment']].copy()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=FIG_SIZE_WIDE)

sns.violinplot(
    data=df_submitters, x='outcome', y='first_score',
    palette=PALETTE_BINARY_LABELS, inner='quartile', ax=ax1,
    order=[LABEL_COMPLETED, LABEL_NOT_COMPLETED],
)
ax1.set_xlabel('')
ax1.set_ylabel('First assessment score')
ax1.set_title('First Assessment Score by Outcome (submitters only)')
sns.despine(ax=ax1)

sns.violinplot(
    data=df_submitters, x='outcome', y='first_submit_day',
    palette=PALETTE_BINARY_LABELS, inner='quartile', ax=ax2,
    order=[LABEL_COMPLETED, LABEL_NOT_COMPLETED],
)
ax2.set_xlabel('')
ax2.set_ylabel('First submission day (relative to course start)')
ax2.set_title('First Submission Timing by Outcome (submitters only)')
sns.despine(ax=ax2)

fig.suptitle('Assessment-Based Early Signals', fontsize=14, y=1.02)
fig.tight_layout()
save_fig(fig, '04_assessment_signal')
plt.show()

> **Interpretazione:**
> - **Consegna vs non-consegna** è di per sé un segnale potente: gli studenti che hanno consegnato almeno una valutazione nei primi 28 giorni hanno un tasso di completamento sostanzialmente più alto.
> - Tra chi ha consegnato, il **punteggio alla prima valutazione** mostra separazione — chi completa tende ad avere punteggi più alti alla prima valutazione.
> - Il **timing di consegna** fornisce un segnale aggiuntivo: una consegna anticipata può indicare una migliore gestione del tempo e un engagement maggiore.
>
> **Avvertenza:** I segnali delle valutazioni hanno alta percentuale di dati mancanti. I risultati del t-test per `first_score` e `first_submit_day` riflettono solo la sottopopolazione di chi ha consegnato. Il segnale di *non-consegna* è catturato dalle metriche di engagement (ghost student/studenti a bassa attività).

## 9. Segnale Ghost Student

Il Notebook 02 ha identificato i "ghost student" — iscrizioni con zero attività VLE nei primi 28 giorni. Qui quantifichiamo questo segnale formalmente usando **intervalli di confidenza bootstrap** per la differenza nel tasso di completamento tra studenti ghost e attivi.

L'approccio bootstrap è usato perché la suddivisione ghost/attivi produce gruppi di dimensioni molto diverse, e il tasso di completamento per i ghost student è vicino allo zero — le assunzioni parametriche per l'IC potrebbero non reggere bene a questi estremi.

In [ ]:
# --- Ghost vs active student completion rate with bootstrap CI ---
# Ghost students = zero VLE activity in first 28 days.
# In the BQ2 dataset, they have active_days_first_28 = 0 (COALESCE'd).
df_signals['is_ghost'] = df_signals['active_days_first_28'] == 0

ghost_completed = df_signals[df_signals['is_ghost']]['completed']
active_completed = df_signals[~df_signals['is_ghost']]['completed']

ghost_rate = ghost_completed.mean() * 100
active_rate = active_completed.mean() * 100

# Bootstrap CI for each group's completion rate
ghost_ci = bootstrap_ci(ghost_completed, statistic_fn=np.mean, n_bootstrap=2000)
active_ci = bootstrap_ci(active_completed, statistic_fn=np.mean, n_bootstrap=2000)

print('=== Ghost vs Active Student Completion Rate ===\n')
print(f'  Ghost students  (n={len(ghost_completed):,}):  '
      f'{ghost_rate:.1f}%  95% CI: [{ghost_ci[0]*100:.1f}%, {ghost_ci[1]*100:.1f}%]')
print(f'  Active students (n={len(active_completed):,}):  '
      f'{active_rate:.1f}%  95% CI: [{active_ci[0]*100:.1f}%, {active_ci[1]*100:.1f}%]')
print(f'\n  Gap: {active_rate - ghost_rate:.1f} percentage points')

# --- Bar chart with bootstrap CI whiskers ---
fig, ax = plt.subplots(figsize=(8, 5))

segments = ['Ghost\n(zero activity)', 'Active\n(>= 1 active day)']
rates = [ghost_rate, active_rate]
ci_lower = [ghost_ci[0] * 100, active_ci[0] * 100]
ci_upper = [ghost_ci[1] * 100, active_ci[1] * 100]
colors_bar = ['#C44E52', '#55A868']

bars = ax.bar(segments, rates, color=colors_bar, edgecolor='white', width=0.5)

# Add CI whiskers
for i, bar in enumerate(bars):
    ax.plot(
        [bar.get_x() + bar.get_width() / 2] * 2,
        [ci_lower[i], ci_upper[i]],
        color='black', linewidth=2,
    )
    # Caps
    cap_width = 0.08
    for y_cap in [ci_lower[i], ci_upper[i]]:
        ax.plot(
            [bar.get_x() + bar.get_width() / 2 - cap_width,
             bar.get_x() + bar.get_width() / 2 + cap_width],
            [y_cap, y_cap],
            color='black', linewidth=2,
        )

# Annotate with rate + CI
for i, bar in enumerate(bars):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        ci_upper[i] + 2,
        f'{rates[i]:.1f}%\n[{ci_lower[i]:.1f}–{ci_upper[i]:.1f}%]',
        ha='center', fontsize=10, color='#333333',
    )

ax.set_ylabel(LABEL_COMPLETION_RATE)
ax.set_title('Completion Rate: Ghost vs. Active Students\n(with 95% Bootstrap CI)')
ax.set_ylim(0, max(ci_upper) + 15)
sns.despine()
fig.tight_layout()
save_fig(fig, '04_ghost_vs_active_completion')
plt.show()

> **Risultato chiave:** Il gap tra ghost student e studenti attivi è enorme e gli intervalli di confidenza bootstrap al 95% non si sovrappongono. I ghost student — quelli con zero attività VLE nei primi 28 giorni — hanno un tasso di completamento prossimo allo zero.
>
> Questo è il **segnale singolo più forte** nel dataset: se uno studente non ha cliccato su nessuna risorsa VLE entro le prime 4 settimane, la sua probabilità di completare il corso è trascurabile.
>
> **Priorità di intervento:** I ghost student sono il frutto più facile da raccogliere. Non hanno bisogno di contenuti migliori — hanno bisogno di essere *attivati*. In termini SaaS, questo è il gap dell'onboarding: utenti che si sono iscritti ma non hanno mai sperimentato il valore del prodotto.

## 10. Conclusioni chiave e prossimi passi

### Cosa abbiamo imparato

1. **Tutti e 8 i segnali precoci mostrano differenze statisticamente significative** tra chi completa e chi non completa. Con ~32K iscrizioni, anche differenze modeste raggiungono la significatività — motivo per cui l'effect size (Cohen's d) è il criterio di classificazione primario.

2. **I predittori comportamentali più forti** sono le metriche di volume di engagement: giorni attivi, click totali e decile di engagement. Questi segnali catturano sia la frequenza che l'intensità dell'interazione con la piattaforma nei primi 28 giorni.

3. **La correzione per confronti multipli conferma la robustezza.** La maggior parte dei segnali rimane significativa dopo correzione sia Bonferroni che Benjamini-Hochberg, indicando che le associazioni sono reali, non artefatti dei test multipli.

4. **Le relazioni dose-response sono monotone.** Più engagement predice coerentemente tassi di completamento più alti in tutti i quartili dei segnali. Non ci sono soglie evidenti o rendimenti decrescenti — la relazione è graduale.

5. **La consegna della valutazione è un predittore binario.** Il fatto che uno studente abbia o meno consegnato una valutazione nei primi 28 giorni è di per sé un segnale potente, indipendente dal punteggio ottenuto.

6. **I ghost student sono il caso estremo.** Zero attività VLE nei primi 28 giorni predice con quasi certezza il non completamento. Questo è il segmento azionabile più chiaro per l'intervento.

7. **Gli effect size sono piccoli-medi** secondo le convenzioni di Cohen. Questo è tipico per dati comportamentali: i singoli segnali spiegano una quota modesta della varianza dell'esito. Il valore pratico viene dalla combinazione di più segnali in un punteggio di engagement (BQ5).

8. **Nessuna affermazione causale.** Tutti i risultati sono associazioni. Gli studenti motivati possono sia impegnarsi di più sia completare di più — l'engagement potrebbe essere un proxy della motivazione, non una causa del successo.

### Cosa viene dopo

| Notebook | Business Question | Focus |
|----------|------------------|-------|
| **05** | BQ3 | Dati demografici vs. comportamento — cosa predice meglio l'esito? |
| **06** | BQ4 | Come influiscono le caratteristiche del corso sulla retention? |
| **07** | BQ5 | Le 3 principali raccomandazioni operative |

---

**Riproducibilità:** Tutte le figure sono salvate in `reports/figures/`. Per riprodurre questo notebook, eseguire prima `python -m run_pipeline`, poi eseguire tutte le celle in ordine.

> **Dai segnali al confronto:** Questo notebook ha identificato *quali* comportamenti precoci predicono il dropout e li ha classificati per effect size. La prossima domanda è: questi segnali comportamentali superano i fattori demografici? O il background di uno studente conta più di ciò che effettivamente fa sulla piattaforma?
>
> Proseguire con il **Notebook 05** (`05_bq3_demographics_vs_behavior.ipynb`) per BQ3: dati demografici vs. comportamento — cosa predice meglio l'esito?